# Policy Gradient, PPO, and Reinforcement Learning for Language Models

This notebook derives, implements, and visualizes the core ideas behind policy gradient methods,
Proximal Policy Optimization (PPO), and how reinforcement learning is applied to train
language models with verifiable rewards.

**Topics covered:**
1. Policy gradient derivation and the log-derivative trick
2. Reward-to-go: why past rewards do not affect current gradients
3. Baselines and variance reduction
4. Importance sampling and the surrogate objective
5. PPO clipping: four-case analysis
6. DAPO / SIPO clipping variants
7. LLMs as MDPs and GRPO-style reward assignment
8. Group-relative baseline estimation
9. End-to-end PPO training loop on a toy task

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Policy Gradient and the Log-Derivative Trick

### Setup

An agent follows a parametric stochastic policy $\pi_\theta(a \mid s)$.  
A trajectory $\tau = (s_0, a_0, s_1, a_1, \ldots, s_{T-1}, a_{T-1})$ is sampled by executing the policy.

The objective is to maximize the expected discounted return:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T-1} \gamma^t r(s_t, a_t)\right]$$

### The challenge

The dependency on $\theta$ lives inside the **sampling distribution**, not inside the reward terms.
Naively swapping $\nabla_\theta$ and $\mathbb{E}$ ignores this dependency entirely.

### Log-derivative trick

For any differentiable distribution $p_\theta$:
$$\nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x)$$

This follows directly from the chain rule applied to $\log$.

### Gradient of the objective

$$\nabla_\theta J(\theta)
= \nabla_\theta \int p_\theta(\tau) R(\tau)\, d\tau
= \int \nabla_\theta p_\theta(\tau) R(\tau)\, d\tau$$

Applying the log-derivative trick:
$$= \int p_\theta(\tau)\, \nabla_\theta \log p_\theta(\tau)\, R(\tau)\, d\tau
= \mathbb{E}_{\tau \sim \pi_\theta}\!\left[\nabla_\theta \log p_\theta(\tau)\; R(\tau)\right]$$

### Decomposing $\log p_\theta(\tau)$

The log-probability of a trajectory factors as:
$$\log p_\theta(\tau) = \log p(s_0) + \sum_{t=0}^{T-1}\left[\log p(s_{t+1}\mid s_t, a_t) + \log \pi_\theta(a_t \mid s_t)\right]$$

The state-transition terms $\log p(s_{t+1}\mid s_t, a_t)$ do not depend on $\theta$, so their gradient is zero.

Therefore:
$$\nabla_\theta \log p_\theta(\tau) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t)$$

### REINFORCE estimator

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[
  \left(\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\right) R(\tau)
\right]$$

**Intuition:** trajectories with high total reward get their log-probabilities maximized more strongly.
The reward $R(\tau)$ acts as a weighting term — good trajectories are reinforced, bad ones are not.

In [ ]:
def softmax(logits):
    e = np.exp(logits - logits.max())
    return e / e.sum()


def log_policy_gradient(logits, action_taken):
    """Gradient of log pi(a|s) with respect to logits."""
    probs = softmax(logits)
    grad = -probs.copy()
    grad[action_taken] += 1.0
    return grad


def reinforce_step(logits, trajectory_rewards, trajectory_actions, lr=0.05):
    """
    Single REINFORCE update over a batch of single-step trajectories.
    Each trajectory is one (action, reward) pair.
    """
    grad_total = np.zeros_like(logits)
    for action, reward in zip(trajectory_actions, trajectory_rewards):
        grad_total += reward * log_policy_gradient(logits, action)
    grad_total /= len(trajectory_actions)
    logits = logits + lr * grad_total
    return logits


n_actions = 4
true_rewards = np.array([1.0, 3.0, 0.5, 2.0])

logits = np.zeros(n_actions)
history = [softmax(logits).copy()]

for _ in range(300):
    probs = softmax(logits)
    actions = np.random.choice(n_actions, size=64, p=probs)
    rewards = true_rewards[actions] + np.random.randn(64) * 0.1
    logits = reinforce_step(logits, rewards, actions, lr=0.1)
    history.append(softmax(logits).copy())

history = np.array(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i in range(n_actions):
    axes[0].plot(history[:, i], label=f'a={i} (r={true_rewards[i]})')
axes[0].set_xlabel('Update step')
axes[0].set_ylabel('Policy probability')
axes[0].set_title('REINFORCE: policy probability over training')
axes[0].legend()

axes[1].bar(range(n_actions), softmax(logits), color='steelblue')
axes[1].set_xlabel('Action')
axes[1].set_ylabel('Final probability')
axes[1].set_title('Final policy (should peak at a=1, highest reward)')
axes[1].set_xticks(range(n_actions))
axes[1].set_xticklabels([f'a={i}\nr={true_rewards[i]}' for i in range(n_actions)])
plt.tight_layout()
plt.show()
print(f'Final policy probabilities: {softmax(logits).round(3)}')

## 2. Reward-to-Go: Why Past Rewards Do Not Matter

### Key identity: zero-gradient of log-probability

For a random variable $a \sim \pi_\theta(\cdot \mid s)$:
$$\mathbb{E}_{a \sim \pi_\theta}\!\left[\nabla_\theta \log \pi_\theta(a \mid s)\right] = 0$$

**Proof:**
$$\mathbb{E}_{a \sim \pi_\theta}\!\left[\nabla_\theta \log \pi_\theta(a \mid s)\right]
= \int \pi_\theta(a \mid s)\, \nabla_\theta \log \pi_\theta(a \mid s)\, da
= \int \nabla_\theta \pi_\theta(a \mid s)\, da
= \nabla_\theta \int \pi_\theta(a \mid s)\, da
= \nabla_\theta\, 1 = 0$$

The last step holds because any valid probability distribution integrates to 1.

### Consequence: past rewards have zero gradient contribution

Expanding the REINFORCE estimator, the full return $R(\tau)$ splits into two sums:
$$R(\tau) = \underbrace{\sum_{t'<t} \gamma^{t'} r_{t'}}_{\text{past rewards}} + \underbrace{\sum_{t'\geq t} \gamma^{t'} r_{t'}}_{\text{future rewards}}
$$

For the past-reward term, applying the law of total expectation and conditioning on
the history $(s_0, a_0, \ldots, s_t)$:
- The past rewards $\sum_{t'<t} r_{t'}$ are **constants** with respect to $a_t$.
- By the Markov property, $a_t \sim \pi_\theta(\cdot \mid s_t)$ depends only on $s_t$.
- Therefore: $\mathbb{E}\!\left[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\cdot c\right] = c\cdot\mathbb{E}\!\left[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\right] = c \cdot 0 = 0$

### Reward-to-go estimator

Only future rewards contribute. Define:
$$\hat{R}_t = \sum_{t'=t}^{T-1} \gamma^{t'-t} r_{t'}$$

The unbiased, lower-variance gradient estimator becomes:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\; \hat{R}_t
\right]$$

**Intuition:** an action at time $t$ cannot retroactively change what happened before $t$.
Weighting by past rewards introduces noise without any signal.

In [ ]:
def compute_rewards_to_go(rewards, gamma=0.99):
    T = len(rewards)
    rtg = np.zeros(T)
    cumulative = 0.0
    for t in reversed(range(T)):
        cumulative = rewards[t] + gamma * cumulative
        rtg[t] = cumulative
    return rtg


def estimate_gradient_variance(use_rtg, n_trials=200, T=20, gamma=0.99):
    """Compare gradient estimate variance: full return vs reward-to-go."""
    grad_estimates = []
    for _ in range(n_trials):
        rewards = np.random.randn(T) + np.linspace(0, 1, T)
        log_grads = np.random.randn(T)

        if use_rtg:
            weights = compute_rewards_to_go(rewards, gamma)
        else:
            weights = np.full(T, rewards.sum())

        grad_estimates.append((log_grads * weights).mean())

    return np.array(grad_estimates)


full_return_grads = estimate_gradient_variance(use_rtg=False)
rtg_grads = estimate_gradient_variance(use_rtg=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(full_return_grads, bins=30, alpha=0.7, color='tomato', label='Full return')
axes[0].hist(rtg_grads, bins=30, alpha=0.7, color='steelblue', label='Reward-to-go')
axes[0].set_xlabel('Gradient estimate')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of gradient estimates')
axes[0].legend()

labels = ['Full return', 'Reward-to-go']
variances = [full_return_grads.var(), rtg_grads.var()]
axes[1].bar(labels, variances, color=['tomato', 'steelblue'])
axes[1].set_ylabel('Variance')
axes[1].set_title('Gradient estimate variance comparison')
for i, v in enumerate(variances):
    axes[1].text(i, v * 1.01, f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()
print(f'Variance (full return):    {full_return_grads.var():.4f}')
print(f'Variance (reward-to-go):   {rtg_grads.var():.4f}')
print(f'Variance reduction factor: {full_return_grads.var() / rtg_grads.var():.2f}x')

## 3. Baselines and Variance Reduction

### Subtracting a baseline leaves the gradient unbiased

Any function $b(s_t)$ that does **not** depend on $a_t$ can be subtracted from $\hat{R}_t$
without introducing bias:

$$\mathbb{E}_{a_t \sim \pi_\theta}\!\left[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\cdot b(s_t)\right]
= b(s_t)\cdot\underbrace{\mathbb{E}_{a_t}\!\left[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\right]}_{=\;0}
= 0$$

Therefore:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\;\bigl(\hat{R}_t - b(s_t)\bigr)
\right]$$

### Why this reduces variance

Suppose rewards in state $s$ always lie in $[10, 11]$ while in another state they lie in $[0, 1]$.
Without a baseline, the gradient weights are $\approx 10$ for the first state and $\approx 0.5$ for the second,
creating large scale differences that inflate variance.

By choosing $b(s)$ close to $\mathbb{E}[\hat{R}_t \mid s_t = s]$:
- The adjusted rewards $\hat{R}_t - b(s_t)$ are centered near zero.
- The scale of the gradient weights is normalized across states.
- The empirical variance of the Monte Carlo gradient estimate decreases.

### Advantage function

The baseline-adjusted reward defines the **advantage**:
$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$$

where $Q(s, a) = \mathbb{E}[\hat{R}_t \mid s_t=s, a_t=a]$ and $V(s) = \mathbb{E}_{a\sim\pi}[Q(s,a)]$.

Positive advantage $\Rightarrow$ action is better than average (reinforce it).  
Negative advantage $\Rightarrow$ action is worse than average (suppress it).

In [ ]:
def estimate_gradient_with_baseline(n_trials=500, n_samples=32):
    """
    Toy single-state, 2-action bandit.
    True rewards: action 0 -> N(2, 1), action 1 -> N(5, 1)
    Compare three baselines: none / constant / state-dependent value.
    """
    true_means = np.array([2.0, 5.0])
    logits = np.array([0.0, 0.0])
    probs = softmax(logits)

    results = {'no_baseline': [], 'constant_baseline': [], 'value_baseline': []}

    constant_b = true_means @ probs
    value_b = true_means @ probs

    for _ in range(n_trials):
        actions = np.random.choice(2, size=n_samples, p=probs)
        rewards = true_means[actions] + np.random.randn(n_samples)

        for method, baseline in [('no_baseline', 0.0),
                                   ('constant_baseline', constant_b),
                                   ('value_baseline', value_b)]:
            adjusted = rewards - baseline
            grad_estimates = []
            for a, adj_r in zip(actions, adjusted):
                g = log_policy_gradient(logits, a)
                grad_estimates.append(adj_r * g[1])
            results[method].append(np.mean(grad_estimates))

    return {k: np.array(v) for k, v in results.items()}


grad_results = estimate_gradient_with_baseline()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = {'no_baseline': 'tomato', 'constant_baseline': 'goldenrod', 'value_baseline': 'steelblue'}
labels = {'no_baseline': 'No baseline', 'constant_baseline': 'Constant baseline',
          'value_baseline': 'Value function baseline'}

for method, vals in grad_results.items():
    axes[0].hist(vals, bins=40, alpha=0.6, color=colors[method], label=labels[method])
axes[0].set_xlabel('Gradient estimate (action 1 logit)')
axes[0].set_ylabel('Count')
axes[0].set_title('Gradient estimate distributions')
axes[0].legend()

method_names = list(grad_results.keys())
variances = [grad_results[m].var() for m in method_names]
bar_colors = [colors[m] for m in method_names]
axes[1].bar([labels[m] for m in method_names], variances, color=bar_colors)
axes[1].set_ylabel('Variance of gradient estimate')
axes[1].set_title('Variance reduction from baselines')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(variances):
    axes[1].text(i, v * 1.01, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()
for m in method_names:
    print(f'{labels[m]:35s}: var = {grad_results[m].var():.4f}')

## 4. Importance Sampling and the Surrogate Objective

### The on-policy bottleneck

REINFORCE is **on-policy**: the gradient estimator requires trajectories sampled from
the current policy $\pi_\theta$. After each gradient step, all previously collected
trajectories become stale and must be discarded. This is sample-inefficient.

### Importance sampling

Importance sampling re-weights samples from an old policy $\pi_{\theta_{\text{old}}}$
to estimate expectations under the new policy $\pi_\theta$:

$$\mathbb{E}_{x \sim p}[f(x)]
= \int p(x) f(x)\, dx
= \int q(x) \frac{p(x)}{q(x)} f(x)\, dx
= \mathbb{E}_{x \sim q}\!\left[\frac{p(x)}{q(x)} f(x)\right]$$

### PPO surrogate objective

Applying importance sampling to the policy gradient, and correcting only the action distribution
(the state distribution shift is acknowledged but not corrected exactly):

$$L^{IS}(\theta) = \mathbb{E}_{s \sim \pi_{\theta_{\text{old}}},\; a \sim \pi_{\theta_{\text{old}}}}
\!\left[\frac{\pi_\theta(a \mid s)}{\pi_{\theta_{\text{old}}}(a \mid s)}\; \hat{A}(s, a)\right]$$

Define the **probability ratio**:
$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

So the surrogate is:
$$L^{IS}(\theta) = \mathbb{E}_t\!\left[r_t(\theta)\; \hat{A}_t\right]$$

When $r_t = 1$ (new policy equals old policy), this coincides with the standard policy gradient.

### Problem with unconstrained IS

If $r_t$ drifts far from 1, the importance weights become large,
the gradient step is oversized, and training destabilizes.
PPO addresses this by **clipping** $r_t$.

In [ ]:
def importance_weight(pi_new, pi_old):
    return pi_new / (pi_old + 1e-8)


pi_old_vals = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 0.9])
pi_new_range = np.linspace(0.01, 0.99, 200)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for pi_old in pi_old_vals:
    ratios = importance_weight(pi_new_range, pi_old)
    axes[0].plot(pi_new_range, ratios, label=f'$\\pi_{{old}}$={pi_old}')

axes[0].axhline(1.0, color='black', linestyle='--', alpha=0.5, label='ratio=1')
axes[0].set_xlabel('$\\pi_{\\theta}(a|s)$')
axes[0].set_ylabel('$r_t = \\pi_\\theta / \\pi_{old}$')
axes[0].set_title('Importance ratio as a function of new policy probability')
axes[0].set_ylim(0, 8)
axes[0].legend(fontsize=8)

ratios_demo = np.linspace(0.1, 4.0, 300)
advantage_pos = 1.0
surr_pos = ratios_demo * advantage_pos
axes[1].plot(ratios_demo, surr_pos, label='$r_t \\cdot \\hat{A}$ (unconstrained, $\\hat{A}>0$)',
             color='steelblue', linewidth=2)
axes[1].axvline(1.0, color='grey', linestyle=':', alpha=0.7)
axes[1].set_xlabel('$r_t(\\theta)$')
axes[1].set_ylabel('Surrogate objective contribution')
axes[1].set_title('Unconstrained IS surrogate grows without bound')
axes[1].legend()

plt.tight_layout()
plt.show()
print('When pi_old=0.1 and pi_new=0.9, importance ratio =', importance_weight(0.9, 0.1).round(2))
print('This 9x weight can cause catastrophically large gradient steps.')

## 5. PPO Clipping: Four-Case Analysis

### The PPO-Clip objective

PPO limits how far $r_t$ can deviate from 1 using a **clipped surrogate**:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[
  \min\!\left(r_t(\theta)\,\hat{A}_t,\;
              \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\,\hat{A}_t\right)
\right]$$

where $\varepsilon \approx 0.2$ is a hyperparameter.

### Four-case analysis

| Case | $\hat{A}_t$ | $r_t$ | Clipped? | Gradient | Intuition |
|------|------------|--------|----------|----------|----------|
| 1 | $> 0$ | $> 1+\varepsilon$ | Yes → flat | Zero | Good action; policy already favors it over old — no need to push further |
| 2 | $> 0$ | $\leq 1+\varepsilon$ | No | $\nabla_\theta r_t\,\hat{A}_t$ | Good action but policy not yet ahead — update normally |
| 3 | $< 0$ | $< 1-\varepsilon$ | Yes → flat | Zero | Bad action; policy already suppresses it — no need to suppress further |
| 4 | $< 0$ | $\geq 1-\varepsilon$ | No | $\nabla_\theta r_t\,\hat{A}_t$ | Bad action but policy still gives it too much probability — update to suppress |

### Why the $\min$ enforces this

For $\hat{A}_t > 0$:
- When $r_t > 1+\varepsilon$: $\text{clip}(r_t,\cdot) = 1+\varepsilon$, so
  $\min(r_t\hat{A}_t, (1+\varepsilon)\hat{A}_t) = (1+\varepsilon)\hat{A}_t$ — a constant w.r.t. $\theta$.
- When $r_t \leq 1+\varepsilon$: both terms agree and the gradient flows normally.

For $\hat{A}_t < 0$:
- When $r_t < 1-\varepsilon$: $\text{clip}(r_t,\cdot) = 1-\varepsilon$, so
  $\min(r_t\hat{A}_t, (1-\varepsilon)\hat{A}_t) = (1-\varepsilon)\hat{A}_t$ — constant.
- When $r_t \geq 1-\varepsilon$: gradient flows normally.

In [ ]:
def ppo_clip_objective(ratio, advantage, eps=0.2):
    clipped = np.clip(ratio, 1 - eps, 1 + eps)
    return np.minimum(ratio * advantage, clipped * advantage)


ratios = np.linspace(0.0, 3.0, 500)
eps = 0.2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, advantage, title_suffix in zip(axes, [1.0, -1.0], ['Positive advantage ($\\hat{A} > 0$)',
                                                              'Negative advantage ($\\hat{A} < 0$)']):
    unclipped = ratios * advantage
    ppo = ppo_clip_objective(ratios, advantage, eps)

    ax.plot(ratios, unclipped, '--', color='tomato', linewidth=1.5, label='Unclipped $r_t \\hat{A}$')
    ax.plot(ratios, ppo, color='steelblue', linewidth=2.5, label='PPO-Clip objective')
    ax.axvline(1 - eps, color='grey', linestyle=':', alpha=0.7, label=f'$1-\\varepsilon$ = {1-eps}')
    ax.axvline(1 + eps, color='black', linestyle=':', alpha=0.7, label=f'$1+\\varepsilon$ = {1+eps}')
    ax.axvline(1.0, color='green', linestyle='-', alpha=0.3, linewidth=1)

    flat_region_color = 'lightyellow'
    if advantage > 0:
        ax.axvspan(1 + eps, 3.0, alpha=0.15, color='tomato', label='No gradient (clipped)')
    else:
        ax.axvspan(0.0, 1 - eps, alpha=0.15, color='tomato', label='No gradient (clipped)')

    ax.set_xlabel('Probability ratio $r_t(\\theta)$')
    ax.set_ylabel('Objective value')
    ax.set_title(f'PPO-Clip: {title_suffix}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Four-case verification:')
test_cases = [
    (2.0,  1.0, 'Case 1: A>0, r>1+eps -> clipped'),
    (0.8,  1.0, 'Case 2: A>0, r<=1+eps -> active gradient'),
    (0.5, -1.0, 'Case 3: A<0, r<1-eps -> clipped'),
    (1.2, -1.0, 'Case 4: A<0, r>=1-eps -> active gradient'),
]
for r, A, desc in test_cases:
    val = ppo_clip_objective(np.array([r]), A, eps)[0]
    unclip_val = r * A
    print(f'  {desc}: obj={val:.3f}, unclipped={unclip_val:.3f}, clipped={val != unclip_val}')

## 6. DAPO and SIPO Clipping Variants

### Motivation for alternatives

PPO's clipping has two behaviors that researchers have questioned:

1. **Complete zero-out**: when outside the clip region, the gradient is exactly zero —
   no learning signal at all, which can slow convergence.
2. **Asymmetric diversity concern**: aggressively suppressing bad actions can collapse
   the policy toward a mode, losing diversity and converging to a local optimum.

### DAPO / GRPO variant

One approach clips the **value** but preserves a **constant gradient** outside the clip region:
- For $\hat{A} > 0$ and $r_t > 1+\varepsilon$: use slope 1 (constant gradient) instead of 0.
- Rationale: the model is doing well but still benefits from learning, just at a reduced effective rate.

### SIPO (Simplified Importance-sampled Policy Optimization)

SIPO applies **symmetric clipping at both ends** regardless of advantage sign:

$$L^{\text{SIPO}}_t = \min\!\left(\text{clip}(r_t, 1/(1+\varepsilon), 1+\varepsilon),\; r_t\right) \cdot \hat{A}_t$$

Key difference: for $\hat{A} < 0$, SIPO **also clips** when $r_t > 1+\varepsilon$,
preventing very large ratios from contributing large negative gradients.

**Philosophy:** large $r_t$ always signals that the policy has changed a lot from the old policy,
regardless of advantage sign. The update magnitude should be bounded to prevent instability,
not just when the sign would otherwise encourage unbounded updates.

In [ ]:
def dapo_objective(ratio, advantage, eps=0.2):
    """DAPO-style: clip value but keep constant gradient outside region."""
    if advantage >= 0:
        return np.where(ratio > 1 + eps,
                        (1 + eps) * advantage + (ratio - (1 + eps)) * 0.0,
                        ratio * advantage)
    else:
        return np.where(ratio < 1 - eps,
                        (1 - eps) * advantage,
                        ratio * advantage)


def sipo_objective(ratio, advantage, eps=0.2):
    """SIPO: clip r_t to [1/(1+eps), 1+eps] unconditionally."""
    lo = 1.0 / (1.0 + eps)
    hi = 1.0 + eps
    clipped_ratio = np.clip(ratio, lo, hi)
    return np.minimum(clipped_ratio, ratio) * advantage


ratios = np.linspace(0.05, 3.5, 600)
eps = 0.2

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for col, advantage in enumerate([1.0, -1.0]):
    ppo_vals = ppo_clip_objective(ratios, advantage, eps)
    dapo_vals = dapo_objective(ratios, advantage, eps)
    sipo_vals = sipo_objective(ratios, advantage, eps)

    sign_str = 'Positive' if advantage > 0 else 'Negative'

    for row, (vals, method) in enumerate([(ppo_vals, 'PPO-Clip'),
                                            (sipo_vals, 'SIPO')]):
        ax = axes[row][col]
        ax.plot(ratios, ratios * advantage, '--', color='lightgrey', linewidth=1.2, label='Unclipped')
        ax.plot(ratios, ppo_vals, color='steelblue', alpha=0.6, linewidth=1.5, label='PPO-Clip')
        ax.plot(ratios, dapo_vals, color='goldenrod', linewidth=1.5, label='DAPO')
        ax.plot(ratios, sipo_vals, color='tomato', linewidth=1.5, label='SIPO')
        ax.axvline(1 - eps, color='grey', linestyle=':', alpha=0.5)
        ax.axvline(1 + eps, color='grey', linestyle=':', alpha=0.5)
        ax.set_xlabel('$r_t$')
        ax.set_ylabel('Objective')
        ax.set_title(f'{sign_str} advantage ($\\hat{{A}}={advantage}$)')
        ax.legend(fontsize=8)
        break

    ax = axes[0][col]
    ax.cla()
    ax.plot(ratios, ratios * advantage, '--', color='lightgrey', linewidth=1.2, label='Unclipped')
    ax.plot(ratios, ppo_vals, color='steelblue', linewidth=2, label='PPO-Clip')
    ax.plot(ratios, dapo_vals, color='goldenrod', linewidth=2, label='DAPO')
    ax.plot(ratios, sipo_vals, color='tomato', linewidth=2, label='SIPO')
    ax.axvline(1 - eps, color='grey', linestyle=':', alpha=0.5)
    ax.axvline(1 + eps, color='grey', linestyle=':', alpha=0.5)
    ax.set_xlabel('$r_t$')
    ax.set_ylabel('Objective')
    ax.set_title(f'{sign_str} advantage ($\\hat{{A}}={advantage:.0f}$)')
    ax.legend(fontsize=8)

    ax2 = axes[1][col]
    ppo_grad = np.gradient(ppo_vals, ratios)
    dapo_grad = np.gradient(dapo_vals, ratios)
    sipo_grad = np.gradient(sipo_vals, ratios)
    ax2.plot(ratios, ppo_grad, color='steelblue', linewidth=2, label='PPO-Clip')
    ax2.plot(ratios, dapo_grad, color='goldenrod', linewidth=2, label='DAPO')
    ax2.plot(ratios, sipo_grad, color='tomato', linewidth=2, label='SIPO')
    ax2.axvline(1 - eps, color='grey', linestyle=':', alpha=0.5)
    ax2.axvline(1 + eps, color='grey', linestyle=':', alpha=0.5)
    ax2.set_xlabel('$r_t$')
    ax2.set_ylabel('$d(\\text{obj})/d(r_t)$')
    ax2.set_title(f'Gradient of objective w.r.t. $r_t$ — {sign_str} advantage')
    ax2.legend(fontsize=8)
    ax2.set_ylim(-3, 3)

plt.tight_layout()
plt.show()
print('PPO zeros gradient outside clip; DAPO retains it; SIPO clips large r symmetrically.')

## 7. LLMs as Markov Decision Processes

### MDP formulation for autoregressive generation

A language model generating a sequence can be viewed as an agent in an MDP:

| MDP Component | Language Model Equivalent |
|--------------|---------------------------|
| State $s_t$ | Prompt $x_{1:L}$ concatenated with all tokens generated so far: $(y_1, \ldots, y_{t-1})$ |
| Action $a_t$ | Next token $y_t$ |
| Policy $\pi_\theta(a_t \mid s_t)$ | $\pi_\theta(y_t \mid x_{1:L}, y_1, \ldots, y_{t-1})$ |
| Transition $p(s_{t+1} \mid s_t, a_t)$ | Deterministic append: $s_{t+1} = (s_t, y_t)$ |
| Reward $R$ | Verifiable correctness of the final answer $y_{1:N}$ |

### Transition dynamics are trivial but the control problem is hard

The state transition is deterministic: appending $y_t$ to the context.
However, the action space is the vocabulary (size $\sim 50{,}000$),
the horizon can be thousands of steps, and the reward is sparse (only at the end).

### Reward structure

For verifiable tasks (math, code, formal reasoning):
$$R(\tau) = \begin{cases} 1 & \text{if the extracted answer matches the ground truth} \\ 0 & \text{otherwise} \end{cases}$$

The reward is applied to the **entire trajectory** as a single terminal signal.
Intermediate tokens (including thinking tokens) receive zero intermediate reward.

### Chain-of-thought as learned latent computation

In the MDP frame, thinking tokens are intermediate actions with no direct reward signal.
RL discovers that certain patterns of thinking tokens — even messy, non-linear ones —
increase the probability of arriving at a correct final answer.
This is fundamentally different from supervised fine-tuning on human-written reasoning chains:
the model learns **what thinking is useful** rather than imitating a prescribed format.

### REINFORCE / PPO objective for LLMs

The probability ratio at each token:
$$r_t(\theta) = \frac{\pi_\theta(y_t \mid x, y_{<t})}{\pi_{\theta_{\text{old}}}(y_t \mid x, y_{<t})}$$

The PPO-Clip objective summed over all generated tokens:
$$L^{\text{LM-PPO}}(\theta) = \mathbb{E}_{\tau}\!\left[
  \sum_{t=1}^{N} \min\!\left(r_t(\theta)\,\hat{A}_t,\;
    \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\,\hat{A}_t\right)
\right]$$

where the advantage uses the group-relative baseline described in the next section.

In [ ]:
def simulate_llm_mdp_episode(vocab_size=50, seq_len=10, n_answer_tokens=2):
    """
    Minimal LLM-as-MDP simulation.
    State: sequence of token indices generated so far.
    Action: next token from a uniform policy.
    Reward: 1 if final n_answer_tokens match target, else 0.
    """
    target_answer = np.random.randint(0, vocab_size, size=n_answer_tokens)
    policy_logits = np.zeros(vocab_size)

    generated = []
    for t in range(seq_len):
        probs = softmax(policy_logits + np.random.randn(vocab_size) * 0.1)
        token = np.random.choice(vocab_size, p=probs)
        generated.append(token)

    answer_tokens = np.array(generated[-n_answer_tokens:])
    reward = float(np.array_equal(answer_tokens, target_answer))

    return {
        'trajectory': generated,
        'answer_tokens': answer_tokens,
        'target': target_answer,
        'reward': reward,
        'thinking_tokens': generated[:-n_answer_tokens]
    }


np.random.seed(42)
n_episodes = 1000
episodes = [simulate_llm_mdp_episode(vocab_size=10, seq_len=8, n_answer_tokens=1)
            for _ in range(n_episodes)]

rewards = np.array([ep['reward'] for ep in episodes])
thinking_lengths = np.array([len(ep['thinking_tokens']) for ep in episodes])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(['Reward=0 (wrong)', 'Reward=1 (correct)'],
            [int((rewards == 0).sum()), int((rewards == 1).sum())],
            color=['tomato', 'steelblue'])
axes[0].set_title('Episode outcome distribution')
axes[0].set_ylabel('Count')

axes[1].hist(thinking_lengths, bins=range(thinking_lengths.min(), thinking_lengths.max() + 2),
             color='goldenrod', alpha=0.8)
axes[1].set_xlabel('Number of thinking tokens')
axes[1].set_ylabel('Count')
axes[1].set_title('Thinking token length (fixed horizon here)')

ax3_data = {'Sparse reward (terminal only)': [0] * 7 + [rewards.mean()],
            'Dense reward (hypothetical)': list(np.linspace(0, rewards.mean(), 8))}
for label, vals in ax3_data.items():
    axes[2].plot(range(8), vals, marker='o', label=label)
axes[2].set_xlabel('Token position in sequence')
axes[2].set_ylabel('Reward signal')
axes[2].set_title('Sparse vs dense reward in LLM training')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f'Fraction of correct episodes: {rewards.mean():.3f}')
print(f'(Chance level with vocab=10, 1 answer token: {1/10:.3f})')

## 8. Group-Relative Baseline Estimation (GRPO)

### The baseline problem in LLM RL

Training a separate value network $V_\phi(s_t)$ requires additional model calls and introduces
a second optimization target. For LLMs with billions of parameters this is expensive.

### Group-relative baseline

Instead, sample $G$ independent trajectories from the **same prompt** $x$:
$$\tau^{(1)}, \tau^{(2)}, \ldots, \tau^{(G)} \sim \pi_{\theta_{\text{old}}}(\cdot \mid x)$$

Compute the group-average reward:
$$\bar{R} = \frac{1}{G} \sum_{i=1}^{G} R(\tau^{(i)})$$

The advantage for trajectory $i$ at token $t$:
$$\hat{A}_t^{(i)} = \frac{R(\tau^{(i)}) - \bar{R}}{\sigma_R + \delta}$$

where $\sigma_R = \text{std}(R^{(1)}, \ldots, R^{(G)})$ normalizes the scale and $\delta$ is a small constant.

### Why this works

- $\bar{R}$ is a function only of the prompt $x$, not of the specific trajectory actions — so it satisfies the baseline condition (it can be subtracted without introducing bias).
- For questions where all $G$ completions get reward 1 (easy) or all get reward 0 (too hard), $\bar{R} = R^{(i)}$ for all $i$, so the advantage is zero and no gradient step is taken — the model neither reinforces nor suppresses behavior it cannot improve on.
- Only on **mixed-outcome questions** (some correct, some wrong) does the baseline produce a useful gradient signal.

### Numerical example

| Trajectory | $R^{(i)}$ | $R^{(i)} - \bar{R}$ |
|------------|-----------|----------------------|
| 1 | 1 | $1 - 1/8 = 7/8$ |
| 2 | 0 | $0 - 1/8 = -1/8$ |
| $\vdots$ | 0 | $-1/8$ |
| 8 | 0 | $-1/8$ |

One successful trajectory is reinforced; the seven unsuccessful ones are mildly penalized.

In [ ]:
def group_relative_advantages(rewards, delta=1e-8):
    """Compute normalized group-relative advantages."""
    mean_r = np.mean(rewards)
    std_r = np.std(rewards)
    return (rewards - mean_r) / (std_r + delta)


def simulate_grpo_batch(n_questions=12, group_size=8, p_correct_easy=0.9,
                         p_correct_hard=0.1, p_correct_mid=0.4):
    """Simulate GRPO advantages across easy, hard, and medium questions."""
    difficulties = ['easy'] * 4 + ['medium'] * 4 + ['hard'] * 4
    p_map = {'easy': p_correct_easy, 'medium': p_correct_mid, 'hard': p_correct_hard}

    all_advantages = []
    all_rewards = []
    for diff in difficulties:
        rewards = np.random.binomial(1, p_map[diff], size=group_size).astype(float)
        advs = group_relative_advantages(rewards)
        all_advantages.append(advs)
        all_rewards.append(rewards)

    return difficulties, all_rewards, all_advantages


np.random.seed(42)
difficulties, all_rewards, all_advantages = simulate_grpo_batch()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_q = len(difficulties)
x_pos = np.arange(n_q)
mean_rewards = [r.mean() for r in all_rewards]
mean_advs = [a.mean() for a in all_advantages]
std_advs = [a.std() for a in all_advantages]

color_map = {'easy': 'steelblue', 'medium': 'goldenrod', 'hard': 'tomato'}
bar_colors = [color_map[d] for d in difficulties]

axes[0].bar(x_pos, mean_rewards, color=bar_colors, alpha=0.8)
axes[0].set_xlabel('Question index')
axes[0].set_ylabel('Group mean reward')
axes[0].set_title('Group mean reward per question')
axes[0].set_xticks(x_pos)
patches = [mpatches.Patch(color=color_map[d], label=d.capitalize()) for d in ['easy', 'medium', 'hard']]
axes[0].legend(handles=patches)

for i, (advs, diff) in enumerate(zip(all_advantages, difficulties)):
    axes[1].scatter([i] * len(advs), advs, alpha=0.6, color=color_map[diff], s=30)
    axes[1].plot([i - 0.3, i + 0.3], [advs.mean(), advs.mean()],
                 color='black', linewidth=2)

axes[1].axhline(0, color='grey', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Question index')
axes[1].set_ylabel('Group-relative advantage')
axes[1].set_title('Advantages per trajectory (dots) + group mean (bar)')
axes[1].legend(handles=patches)

plt.tight_layout()
plt.show()

print('Mean absolute advantage by difficulty:')
for diff in ['easy', 'medium', 'hard']:
    indices = [i for i, d in enumerate(difficulties) if d == diff]
    mean_abs = np.mean([np.abs(all_advantages[i]).mean() for i in indices])
    print(f'  {diff:8s}: {mean_abs:.3f}')
print('Medium questions carry the most useful gradient signal.')

## 9. End-to-End PPO Training Loop on a Toy Bandit Task

### Task description

A multi-arm bandit with 8 actions and fixed but unknown reward means.
The policy is a softmax over a learnable logit vector.
Training uses the PPO-Clip objective with group-relative advantages.

### Training protocol

At each iteration:
1. Sample $G$ trajectories from the current (old) policy.
2. Compute rewards and group-relative advantages.
3. Compute probability ratios $r_t = \pi_\theta(a) / \pi_{\theta_{\text{old}}}(a)$ for each sampled action.
4. Compute the PPO-Clip objective and take a gradient step.
5. Update the old policy to the current policy.

This directly mirrors how PPO is applied to LLM fine-tuning, with the policy network
replaced by a softmax over logits for tractability.

In [ ]:
def ppo_bandit_training(n_actions=8, n_iterations=400, group_size=16,
                         lr=0.15, eps=0.2, n_ppo_epochs=4):
    np.random.seed(42)

    true_rewards = np.array([0.2, 0.5, 0.9, 0.3, 0.7, 0.1, 0.6, 0.4])
    best_action = np.argmax(true_rewards)

    logits = np.zeros(n_actions)

    reward_history = []
    prob_best_history = []
    entropy_history = []

    for iteration in range(n_iterations):
        pi_old = softmax(logits.copy())

        actions = np.random.choice(n_actions, size=group_size, p=pi_old)
        rewards = true_rewards[actions] + np.random.randn(group_size) * 0.05

        advantages = group_relative_advantages(rewards)

        for _ in range(n_ppo_epochs):
            pi_new = softmax(logits)

            grad = np.zeros(n_actions)
            for a, adv in zip(actions, advantages):
                ratio = pi_new[a] / (pi_old[a] + 1e-8)
                clipped_ratio = np.clip(ratio, 1 - eps, 1 + eps)

                if adv >= 0:
                    effective = min(ratio, clipped_ratio) * adv
                else:
                    effective = max(ratio, clipped_ratio) * adv

                d_ratio_d_logits = np.zeros(n_actions)
                d_ratio_d_logits[a] = pi_new[a] * (1 - pi_new[a]) / (pi_old[a] + 1e-8)
                for j in range(n_actions):
                    if j != a:
                        d_ratio_d_logits[j] = -pi_new[a] * pi_new[j] / (pi_old[a] + 1e-8)

                is_clipped = (ratio > 1 + eps and adv > 0) or (ratio < 1 - eps and adv < 0)
                if not is_clipped:
                    grad += adv * d_ratio_d_logits

            logits += lr * grad / group_size

        pi_curr = softmax(logits)
        mean_r = (pi_curr * true_rewards).sum()
        entropy = -np.sum(pi_curr * np.log(pi_curr + 1e-8))

        reward_history.append(mean_r)
        prob_best_history.append(pi_curr[best_action])
        entropy_history.append(entropy)

    return reward_history, prob_best_history, entropy_history, softmax(logits), true_rewards


reward_hist, prob_best_hist, entropy_hist, final_policy, true_rewards = ppo_bandit_training()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0][0].plot(reward_hist, color='steelblue')
axes[0][0].axhline(true_rewards.max(), color='tomato', linestyle='--', label=f'Optimal = {true_rewards.max()}')
axes[0][0].set_xlabel('Iteration')
axes[0][0].set_ylabel('Expected reward')
axes[0][0].set_title('Expected reward under learned policy')
axes[0][0].legend()

axes[0][1].plot(prob_best_hist, color='goldenrod')
axes[0][1].axhline(1.0, color='grey', linestyle='--', alpha=0.5)
axes[0][1].set_xlabel('Iteration')
axes[0][1].set_ylabel(f'P(best action = {np.argmax(true_rewards)})')
axes[0][1].set_title('Probability assigned to best action')

axes[1][0].plot(entropy_hist, color='tomato')
axes[1][0].set_xlabel('Iteration')
axes[1][0].set_ylabel('Policy entropy')
axes[1][0].set_title('Policy entropy over training')

n_actions = len(true_rewards)
x = np.arange(n_actions)
width = 0.35
axes[1][1].bar(x - width / 2, true_rewards, width, label='True reward', color='steelblue', alpha=0.7)
axes[1][1].bar(x + width / 2, final_policy, width, label='Final policy prob', color='goldenrod', alpha=0.7)
axes[1][1].set_xlabel('Action')
axes[1][1].set_ylabel('Value')
axes[1][1].set_title('True rewards vs final policy probabilities')
axes[1][1].set_xticks(x)
axes[1][1].legend()

plt.tight_layout()
plt.show()

print(f'Final expected reward:   {reward_hist[-1]:.4f}')
print(f'Optimal expected reward: {true_rewards.max():.4f}')
print(f'Final policy probabilities: {final_policy.round(3)}')
print(f'Best action index: {np.argmax(true_rewards)} '
      f'| final policy probability: {final_policy[np.argmax(true_rewards)]:.3f}')